# 03b — SHAP Dependence Plot và nhóm đặc trưng cho reason codes

**Mục đích:** Phân tích quan hệ giữa giá trị đặc trưng và đóng góp SHAP cho Top 9 biến quan trọng nhất của XGBoost, phát hiện tương tác giữa các biến, và kiểm chứng định nghĩa nhóm đặc trưng dùng cho reason codes (Tuần 2 – T4). Notebook tiếp nối `03_shap.ipynb` (SHAP Global) và dùng cùng thiết lập: mô hình `xgboost_default` fit trên Train, mẫu giải thích 3.000 dòng từ Valid (`random_state = 42`), thang giá trị SHAP là log-odds (margin).

**Tài liệu tham chiếu:** `reports/project_plan.md` (mục 3.5e), `configs/config.yaml` (khóa `feature_groups`), `src/explain.py`.

**Phương pháp:**

1. Giá trị SHAP của các cột one-hot được cộng về biến gốc (tính cộng tính được bảo toàn).
2. Tương tác được đo bằng `shap_interaction_values` của `TreeExplainer`: với mỗi cặp biến $(v, w)$, cường độ tương tác là trung bình trị tuyệt đối của tổng các phần tử ma trận tương tác thuộc cặp đó, nhân 2 (do ma trận đối xứng). Phần đường chéo là hiệu ứng chính của từng biến.
3. Mỗi Dependence Plot tô màu theo biến có tương tác mạnh nhất với biến đang xét.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.base import clone

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.config import load_config  # noqa: E402
from src.data_split import load_split_data  # noqa: E402
from src.explain import (  # noqa: E402
    aggregate_by_group,
    aggregate_by_variable,
    check_groups_cover,
    load_feature_groups,
    model_variables_from_freeze,
    source_variable,
    variable_to_group,
)
from src.features import build_features  # noqa: E402
from src.pipelines import make_pipeline  # noqa: E402
from src.train import get_advanced_default_models  # noqa: E402

SEED = load_config()["random_state"]
N_SAMPLE = 3000
TOP_N = 9
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 140)

## 1. Mô hình, mẫu giải thích và giá trị SHAP

Các bước này lặp lại `03_shap.ipynb` (mục 1–3) để notebook có thể chạy độc lập. Bên cạnh giá trị SHAP, `shap_interaction_values` được tính trên cùng mẫu (ma trận 3.000 × 119 × 119).

In [ ]:
train_df, valid_df, _ = load_split_data()
X_train, y_train = build_features(train_df)
X_valid, y_valid = build_features(valid_df)

pipe = make_pipeline(clone(get_advanced_default_models(SEED)["xgboost_default"])).fit(X_train, y_train)
prep, model = pipe[:-1], pipe[-1]
col_tf = pipe.named_steps["preprocess"].named_steps["preprocessor"]

X_sample = X_valid.sample(N_SAMPLE, random_state=SEED)
X_model = prep.transform(X_sample)
columns = [n.split("__", 1)[1] for n in col_tf.get_feature_names_out()]
cat_vars = list(col_tf.transformers_[0][2])
num_cols = list(col_tf.transformers_[1][2])
n_cat_out = len(columns) - len(num_cols)

# Giá trị gốc (chưa chuẩn hóa) của các cột số, phục vụ trục hoành của Dependence Plot.
X_display = pd.DataFrame(X_model.copy(), columns=columns, index=X_sample.index)
X_display[num_cols] = col_tf.named_transformers_["numeric"]["scaler"].inverse_transform(X_model[:, n_cat_out:])

explainer = shap.TreeExplainer(model)
t0 = time.perf_counter()
phi = explainer.shap_values(X_model)
phi_int = explainer.shap_interaction_values(X_model)
print(f"SHAP: {phi.shape} | interaction: {phi_int.shape} | thời gian: {time.perf_counter() - t0:.1f} s")

margin = model.predict(X_model, output_margin=True)
print(f"Sai số cộng tính (SHAP): {np.abs(explainer.expected_value + phi.sum(1) - margin).max():.2e}")
print(f"Sai số cộng tính (interaction): {np.abs(explainer.expected_value + phi_int.sum((1, 2)) - margin).max():.2e}")

## 2. Gộp theo biến gốc và đo cường độ tương tác

Biến phân loại (`PAY_1`–`PAY_6`, `EDUCATION`, `MARRIAGE`, `AGE_BIN`) được gộp từ các cột one-hot về một biến. Với ma trận tương tác, việc gộp thực hiện trên cả hai chiều bằng ma trận chỉ thị cột → biến.

In [ ]:
shap_by_var = aggregate_by_variable(phi, columns, cat_vars)
variables = list(shap_by_var.columns)

# Ma trận tương tác theo biến: M[v, w] = sum_{a thuộc v, b thuộc w} phi_int[:, a, b]

var_of_col = np.array([source_variable(c, cat_vars) for c in columns])
indicator = np.stack([(var_of_col == v) for v in variables], axis=1).astype(float)  # (119, 49)
phi_int_var = np.einsum("nab,av,bw->nvw", phi_int, indicator, indicator)

assert np.allclose(phi_int_var.sum(2), shap_by_var.to_numpy(), atol=1e-4)  # tổng theo hàng = SHAP theo biến

importance = shap_by_var.abs().mean().sort_values(ascending=False)
top_vars = list(importance.index[:TOP_N])
idx = {v: i for i, v in enumerate(variables)}

strength = pd.DataFrame(
    np.abs(phi_int_var).mean(0) * 2, index=variables, columns=variables
)
np.fill_diagonal(strength.values, 0.0)  # chỉ giữ phần tương tác chéo giữa hai biến khác nhau

pairs = (
    strength.where(np.triu(np.ones(strength.shape, dtype=bool), k=1))
    .stack()
    .sort_values(ascending=False)
    .rename("mean_abs_interaction")
    .to_frame()
)
pairs.index.names = ["biến 1", "biến 2"]
pairs.to_csv(FIG_DIR / "shap_xgb_interaction_pairs.csv")
pairs.head(15)

Bảng dưới đây so sánh hiệu ứng chính (đường chéo) và tổng cường độ tương tác của từng biến trong Top 9, cùng đối tác tương tác mạnh nhất.

In [ ]:
main_effect = pd.Series({v: np.abs(phi_int_var[:, idx[v], idx[v]]).mean() for v in variables})
partner = strength.idxmax(axis=1)

summary = pd.DataFrame({
    "mean|SHAP|": importance[top_vars],
    "hiệu ứng chính": main_effect[top_vars],
    "tương tác (tổng)": strength.loc[top_vars].sum(axis=1),
    "đối tác mạnh nhất": partner[top_vars],
    "cường độ": [strength.loc[v, partner[v]] for v in top_vars],
})
summary["tương tác / hiệu ứng chính"] = summary["tương tác (tổng)"] / summary["hiệu ứng chính"]
summary

## 3. Dependence Plot cho Top 9 biến

Trục hoành là giá trị đặc trưng, trục tung là đóng góp SHAP của biến (đã gộp one-hot, thang log-odds). Màu thể hiện giá trị của biến tương tác mạnh nhất. Đường đen là trung vị của SHAP theo phân vị của giá trị đặc trưng, giúp quan sát xu hướng; với biến phân loại `PAY_1` trục hoành là mã trạng thái gốc.

In [ ]:
def raw_value(var):
    # Giá trị gốc dùng để vẽ: cột số lấy từ X_display; PAY_1 lấy từ cột PAY_0 của dữ liệu thô.
    if var in X_display.columns:
        return X_display[var].to_numpy()
    if var == "PAY_1":
        return X_sample["PAY_0"].to_numpy()
    return X_sample[var].to_numpy()


def trend(x, y, n_bins=15):
    # Trung vị của y theo phân vị của x; với x rời rạc ít giá trị thì theo từng giá trị.
    df = pd.DataFrame({"x": x, "y": y})
    if df["x"].nunique() <= n_bins:
        g = df.groupby("x")["y"].median()
        return g.index.to_numpy(), g.to_numpy()
    df["b"] = pd.qcut(df["x"], n_bins, duplicates="drop")
    g = df.groupby("b", observed=True).agg(x=("x", "median"), y=("y", "median"))
    return g["x"].to_numpy(), g["y"].to_numpy()


fig, axes = plt.subplots(3, 3, figsize=(17, 13))
for ax, var in zip(axes.ravel(), top_vars):
    x, y = raw_value(var), shap_by_var[var].to_numpy()
    p = partner[var]
    c = raw_value(p)
    sc = ax.scatter(x, y, c=c, cmap="coolwarm", s=7, alpha=0.7, vmin=np.percentile(c, 2), vmax=np.percentile(c, 98))
    tx, ty = trend(x, y)
    ax.plot(tx, ty, color="black", lw=1.6)
    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.set_xlabel(var)
    ax.set_ylabel("SHAP (log-odds)")
    ax.set_title(f"{var} (hạng {top_vars.index(var) + 1})")
    fig.colorbar(sc, ax=ax, label=p)
fig.suptitle("SHAP Dependence Plot — XGBoost, Top 9 biến (màu: đối tác tương tác mạnh nhất)", y=1.0)
fig.tight_layout()
fig.savefig(FIG_DIR / "shap_xgb_dependence_top9.png", dpi=130, bbox_inches="tight")
plt.show()

### 3.1. Tương tác theo cặp

Ma trận cường độ tương tác giữa các biến trong Top 9 và các biến còn lại có tương tác đáng kể nhất; màu đậm thể hiện tương tác mạnh.

In [ ]:
cols_show = list(dict.fromkeys(top_vars + list(pairs.reset_index().head(12)[["biến 1", "biến 2"]].to_numpy().ravel())))
sub = strength.loc[cols_show, cols_show]

fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(sub.to_numpy(), cmap="Reds")
ax.set_xticks(range(len(cols_show)), cols_show, rotation=60, ha="right")
ax.set_yticks(range(len(cols_show)), cols_show)
fig.colorbar(im, label="mean(|tương tác|)")
ax.set_title("Cường độ tương tác SHAP giữa các biến")
fig.tight_layout()
fig.savefig(FIG_DIR / "shap_xgb_interaction_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()

## 4. Nhóm đặc trưng cho reason codes

Nhóm đặc trưng được khai báo trong `configs/config.yaml` (khóa `feature_groups`), gồm năm nhóm theo `project_plan.md` mục 3.5e. Hàm `check_groups_cover` xác nhận mọi biến trong `configs/feature_freeze_v1.yaml` có đúng một nhóm và không có biến thừa; kiểm tra này sẽ báo lỗi nếu bộ đặc trưng thay đổi sau khi lọc biến ở T4.

In [ ]:
groups = load_feature_groups()
check_groups_cover(groups, model_variables_from_freeze())
assert set(variables) == set(variable_to_group(groups))  # khớp với biến thực tế của mô hình

group_table = pd.DataFrame({
    "nhóm": [spec["label"] for spec in groups.values()],
    "số biến": [len(spec["variables"]) for spec in groups.values()],
    "biến": [", ".join(spec["variables"]) for spec in groups.values()],
}).set_index("nhóm")
group_table

### 4.1. Mức độ quan trọng theo nhóm

Giá trị SHAP của các biến trong cùng nhóm được cộng trước khi lấy trị tuyệt đối, do đó phản ánh đóng góp ròng của nhóm cho từng khách hàng — đúng đại lượng sẽ hiển thị trong reason codes. Cột `Σ mean|SHAP| từng biến` là cách tính khác (cộng mức quan trọng của từng biến), cho thấy mức độ triệt tiêu lẫn nhau giữa các biến cùng nhóm.

In [ ]:
shap_by_group = aggregate_by_group(shap_by_var, groups)

# Kiểm tra cộng tính ở cấp nhóm: base + tổng các nhóm = margin
err = np.abs(explainer.expected_value + shap_by_group.sum(axis=1) - margin).max()
print(f"Sai số cộng tính cấp nhóm: {err:.2e}")
assert err < 1e-3

group_importance = shap_by_group.abs().mean().sort_values(ascending=False)
label_of = {key: spec["label"] for key, spec in groups.items()}
sum_individual = pd.Series({
    label_of[k]: importance[spec["variables"]].sum() for k, spec in groups.items()
})
table = pd.DataFrame({
    "mean|SHAP| nhóm": group_importance,
    "share": group_importance / group_importance.sum(),
    "Σ mean|SHAP| từng biến": sum_individual[group_importance.index],
})
table["tỷ lệ triệt tiêu"] = 1 - table["mean|SHAP| nhóm"] / table["Σ mean|SHAP| từng biến"]
table.to_csv(FIG_DIR / "shap_xgb_importance_by_group.csv")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(table.index[::-1], table["mean|SHAP| nhóm"][::-1], color="#d6336c")
for i, v in enumerate(table["mean|SHAP| nhóm"][::-1]):
    ax.text(v + 0.005, i, f"{v:.2f}", va="center")
ax.set_xlabel("mean(|SHAP nhóm|), log-odds")
ax.set_title("Mức độ quan trọng theo nhóm đặc trưng — XGBoost")
fig.tight_layout()
fig.savefig(FIG_DIR / "shap_xgb_bar_by_group.png", dpi=150, bbox_inches="tight")
plt.show()
table

### 4.2. Độ ổn định và kiểm tra tương quan trong nhóm

Hai kiểm tra bổ sung: (a) thứ hạng nhóm trên các mẫu khác nhau; (b) tương quan Spearman giữa đóng góp SHAP của các biến trong cùng nhóm với đóng góp của nhóm — đo mức độ các biến trong nhóm cùng hướng với nhau, qua đó đánh giá nhóm có đồng nhất hay không.

In [ ]:
def group_importance_for(X_raw):
    phi_x = explainer.shap_values(prep.transform(X_raw))
    by_var = aggregate_by_variable(phi_x, columns, cat_vars)
    return aggregate_by_group(by_var, groups).abs().mean()


ref = group_importance
rows = {}
for label, X_raw in {
    "Valid, seed 0": X_valid.sample(N_SAMPLE, random_state=0),
    "Valid, seed 1": X_valid.sample(N_SAMPLE, random_state=1),
    "Train, seed 42": X_train.sample(N_SAMPLE, random_state=SEED),
}.items():
    other = group_importance_for(X_raw)[ref.index]
    rows[label] = {"Spearman": ref.corr(other, method="spearman"),
                   "cùng thứ tự": bool((other.sort_values(ascending=False).index == ref.index).all())}
display(pd.DataFrame(rows).T)

# (b) Đồng hướng giữa biến và nhóm
coh = {}
for key, spec in groups.items():
    g = shap_by_group[spec["label"]]
    for v in spec["variables"]:
        coh[v] = {"nhóm": spec["label"], "Spearman với nhóm": shap_by_var[v].corr(g, method="spearman"),
                  "mean|SHAP|": importance[v]}
coh = pd.DataFrame(coh).T
coh["Spearman với nhóm"] = coh["Spearman với nhóm"].astype(float)
coh["mean|SHAP|"] = coh["mean|SHAP|"].astype(float)
coh.groupby("nhóm").agg(
    **{"Spearman nhỏ nhất": ("Spearman với nhóm", "min"), "Spearman trung vị": ("Spearman với nhóm", "median")}
)

## 5. Nhận xét

### 5.1. Dạng quan hệ giữa đặc trưng và đóng góp SHAP

- **`PAY_MAX` (hạng 1): quan hệ bậc thang, bão hòa từ mức trễ 2 tháng.** SHAP trung vị ≈ −0,5 khi `PAY_MAX` ≤ 0, tăng lên ≈ +0,3 tại mức 1, đạt ≈ +0,9 đến +1,1 tại mức 2–3 và gần như không tăng thêm với mức cao hơn. Mô hình phân biệt chủ yếu giữa "chưa từng trễ hạn nghiêm trọng" và "đã trễ từ 2 tháng trở lên", không phân biệt mức độ trễ sâu hơn.
- **`PAY_1` (hạng 2): không đơn điệu.** Trạng thái trễ 2 và 3 tháng có SHAP trung vị ≈ +0,7 đến +0,8, nhưng từ 4 tháng trở lên SHAP về ≈ −0,1. Đây không phản ánh rủi ro thực tế thấp đi: các mức này hiếm trong mẫu (các cột one-hot `PAY_1` từ mức 4 trở lên có mean(|SHAP|) = 0, xem `03_shap.ipynb`) và phần đóng góp nhiều khả năng đã được `PAY_MAX` hấp thụ. Đây là biểu hiện của tương quan giữa hai biến, không phải tín hiệu nghiệp vụ.
- **Nhóm số tiền trả (`PAY_AMT1`, `PAY_AMT2`, `PAY_AMT6`) và `PAY_RATIO_5`: quan hệ nghịch, giảm nhanh ở vùng giá trị thấp rồi bão hòa.** SHAP giảm từ khoảng +0,3 (không trả hoặc trả rất ít) xuống −0,4 đến −1,0 ở các khoản trả lớn. `PAY_RATIO_5` phân hóa mạnh trong khoảng 0–2: tỷ lệ gần 0 cho SHAP đến +0,6, tỷ lệ từ 1 trở lên cho SHAP âm sâu. Một quan sát có `PAY_RATIO_5` ≈ 25 là ngoại lệ, đoạn đuôi của đường xu hướng không đủ dữ liệu để kết luận.
- **`BILL_AMT1`: quan hệ không đơn điệu, phụ thuộc vào mức sử dụng hạn mức.** Ở cùng mức dư nợ, khách có `UTIL_1` cao (màu đỏ) nhận SHAP dương, khách có `UTIL_1` thấp (màu xanh) nhận SHAP âm. Cùng một dư nợ mang ý nghĩa rủi ro khác nhau tùy hạn mức, do đó `BILL_AMT1` không nên được diễn giải độc lập với `UTIL_1`/`LIMIT_BAL`.
- **`BILL_DELTA`: xu hướng giảm nhẹ khi biến động dư nợ tăng**, từ ≈ +0,2 ở vùng gần 0 xuống ≈ 0 đến −0,1. Ở vùng giá trị thấp, độ phân tán rất lớn (SHAP từ −0,45 đến +0,8), cho thấy giá trị thấp của biến này gộp nhiều tình huống khác nhau (ví dụ dư nợ luôn bằng 0 và dư nợ ổn định ở mức cao). Chiều tác động cần được diễn giải thận trọng.

### 5.2. Tương tác giữa các đặc trưng

- **Tương tác nổi bật nhất là cặp `PAY_1` – `PAY_MAX`** (cường độ 0,233), gấp hơn 3 lần cặp xếp thứ hai (`PAY_MAX` – `PAY_MEAN`, 0,076). Tại `PAY_MAX` = 2, SHAP trải từ ≈ +0,3 đến ≈ +1,7 tùy theo `PAY_1`: khách có trạng thái tháng gần nhất vẫn trễ (màu đỏ) bị cộng thêm rủi ro đáng kể so với khách đã trả nợ trở lại. Điều này phù hợp với diễn giải nghiệp vụ (trễ hạn còn kéo dài rủi ro hơn trễ hạn đã được xử lý).
- **Phần lớn cặp tương tác mạnh là các cặp biến trùng lặp thông tin trong cùng một nhóm**: `BILL_DELTA` – `BILL_STD` (0,074), `UTIL_2`/`UTIL_3`/`UTIL_4`/`UTIL_6` – `UTIL_MEAN` (0,043–0,069), `BILL_AMT1` – `BILL_AMT2` (0,059), `PAY_MAX` – `PAY_MEAN`. Cường độ tương tác ở đây phản ánh việc mô hình chia sẻ đóng góp giữa các biến tương quan, không phải tương tác nghiệp vụ độc lập.
- **Tương tác mang ý nghĩa nghiệp vụ rõ ràng giữa các nhóm khác nhau có cường độ thấp hơn nhiều**: `BILL_AMT1` – `UTIL_1` (0,072), `BILL_DELTA` – `PAY_AMT3` (0,052), `BILL_AMT1` – `PAY_AMT1` (0,042). Với `PAY_AMT1`, `PAY_AMT2`, `PAY_AMT3`, `PAY_AMT6` và `PAY_RATIO_5`, đối tác mạnh nhất chỉ đạt 0,036–0,052, so với hiệu ứng chính 0,08–0,16.
- **Lưu ý khi đọc cột "tương tác / hiệu ứng chính":** cột tổng tương tác cộng cường độ với 48 biến còn lại, trong đó phần lớn là các số hạng nhỏ; tỷ lệ này (1,4 đến 10,7) chủ yếu cho thấy tương tác phân tán trên nhiều biến, không nên đọc là tương tác chiếm ưu thế so với hiệu ứng chính. Tương tác tập trung thực sự chỉ ở cặp `PAY_1` – `PAY_MAX`.

### 5.3. Nhóm đặc trưng cho reason codes

Năm nhóm (cấu hình tại `configs/config.yaml`, khóa `feature_groups`) bao phủ đủ 49 biến của feature freeze v1, mỗi biến thuộc đúng một nhóm. Cộng tính cấp nhóm được bảo toàn (sai số 3,3 × 10⁻⁶).

| Nhóm | Số biến | mean(\|SHAP nhóm\|) | Tỷ trọng |
|---|---|---|---|
| Lịch sử trễ hạn | 11 | 0,824 | 38,5% |
| Hành vi trả nợ | 19 | 0,515 | 24,1% |
| Mức sử dụng hạn mức | 8 | 0,357 | 16,7% |
| Dư nợ và biến động dư nợ | 8 | 0,329 | 15,4% |
| Nhân khẩu học | 3 | 0,114 | 5,3% |

- **Thứ hạng nhóm ổn định**: trên ba mẫu khác (hai mẫu Valid, một mẫu Train), thứ tự năm nhóm không đổi (Spearman = 1,00).
- **Cộng SHAP trước khi lấy trị tuyệt đối làm tăng tỷ trọng của nhóm trễ hạn** (38,5%, so với khoảng 29% khi cộng mức quan trọng của từng biến). Các biến trễ hạn đóng góp cùng chiều với nhau nên ít triệt tiêu (tỷ lệ triệt tiêu 29%), trong khi nhóm hành vi trả nợ và nhóm dư nợ triệt tiêu khoảng 58% do các biến trong nhóm thường mang dấu ngược nhau trên cùng một khách hàng.
- **Mức đồng hướng giữa biến và nhóm thấp ở nhiều nhóm** (Spearman trung vị giữa SHAP của biến và SHAP của nhóm: 0,20 với nhóm trễ hạn, 0,25 với hành vi trả nợ, 0,38 với dư nợ, 0,55 với hạn mức, 0,59 với nhân khẩu học; giá trị nhỏ nhất của bốn nhóm đầu đều ≤ 0,054). Đóng góp ròng của nhóm vì vậy không thể suy ra từ một biến đơn lẻ; reason codes nên được tính trên SHAP ròng của nhóm, đúng như thiết kế tại `project_plan.md` mục 3.5e.

### 5.4. Giới hạn

- Kết quả áp dụng cho XGBoost tham số mặc định; tuning và monotonic constraints (Tuần 2 – T4/T5) có thể làm thay đổi dạng quan hệ, đặc biệt các điểm không đơn điệu của `PAY_1` và `BILL_AMT1`. Notebook cần được chạy lại trên mô hình cuối.
- Đường xu hướng và Dependence Plot kém tin cậy ở vùng đuôi do ít quan sát (ví dụ `BILL_AMT1` > 600.000, `PAY_RATIO_5` > 5, `PAY_1` ≥ 4). Cường độ tương tác tính trên thang margin của mô hình chưa hiệu chuẩn; hệ số hiệu chuẩn Platt chỉ nhân toàn bộ giá trị với một hằng số nên không đổi thứ hạng.
- Chi phí tính toán của `shap_interaction_values` cho 3.000 dòng × 119 cột là khoảng 220 giây, cao hơn nhiều so với SHAP thường (dưới 1 giây); không phù hợp tính trực tiếp trong demo.
- Nhóm Nhân khẩu học (AGE_BIN, EDUCATION, MARRIAGE) đóng góp thấp (5,3%) nhưng là nhóm liên quan đến nhóm đối tượng được bảo vệ; việc có hiển thị nhóm này trong reason codes gửi tới khách hàng cần được thống nhất cùng kết quả fairness (thành viên A, Tuần 3 – T3).

### 5.5. Đầu vào cho các bước tiếp theo

- **Reason codes (Tuần 3 – T3):** dùng `src.explain.aggregate_by_group` trên SHAP đã nhân hệ số Platt; chọn k nhóm có SHAP ròng dương lớn nhất.
- **Lọc đặc trưng (thành viên A, T4):** các cặp trùng lặp (`BILL_DELTA`/`BILL_STD`, `UTIL_*`/`UTIL_MEAN`, `PAY_MAX`/`PAY_MEAN`) là ứng viên khi loại biến tương quan cao. Nếu bộ đặc trưng thay đổi, cần cập nhật `feature_groups`; `check_groups_cover` (và `tests/test_explain.py`) sẽ báo lỗi khi cấu hình nhóm không còn khớp với `feature_freeze_v1.yaml`.
- **Tuning (thành viên B, T4):** các điểm không đơn điệu của `PAY_1` là cơ sở để cân nhắc monotonic constraints cho `PAY_1`–`PAY_6` và `PAY_MAX`.
- **SHAP Local (Tuần 2 – T5):** lấy các hồ sơ có `PAY_MAX` = 2 nhưng `PAY_1` khác nhau làm ca minh họa cho tương tác chính.

## Hình và bảng xuất ra

Lưu tại `reports/figures/` (không commit, sinh lại bằng cách chạy notebook):

- `shap_xgb_dependence_top9.png` — Dependence Plot Top 9 biến.
- `shap_xgb_interaction_heatmap.png` — ma trận cường độ tương tác.
- `shap_xgb_interaction_pairs.csv` — cường độ tương tác theo cặp biến.
- `shap_xgb_bar_by_group.png`, `shap_xgb_importance_by_group.csv` — mức độ quan trọng theo nhóm đặc trưng.